<a href="https://colab.research.google.com/github/Lilwenz/Lilwenz/blob/Thermal-Error-Prediction/PyTorch_%E6%A8%A1%E5%9E%8B%E5%AE%9E%E7%8E%B0%EF%BC%9A%E6%9C%BA%E5%BA%8A%E7%83%AD%E8%AF%AF%E5%B7%AE%E9%A2%84%E6%B5%8B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import math
from soft_dtw_pytorch import SoftDTW # 用于可微分的 DTW 损失

# --- 位置编码模块 ---
class PositionalEncoding(nn.Module):
    """
    标准的正弦/余弦位置编码
    """
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).transpose(0, 1) # 变为 [max_len, 1, d_model]
        self.register_buffer('pe', pe)

    def forward(self, x):
        """
        Args:
            x: 输入张量，形状为 [序列长度, 批次大小, 特征维度] (T, B, C)
        Returns:
            带有位置编码的输出张量
        """
        # x 的形状是 [T, B, C], pe 的形状是 [max_len, 1, C]
        # 我们只需要 pe 的前 T 个时间步
        x = x + self.pe[:x.size(0), :]
        return x

# --- 时间卷积网络 (TCN) 模块 ---
class Chomp1d(nn.Module):
    """移除序列末尾的额外填充"""
    def __init__(self, chomp_size):
        super(Chomp1d, self).__init__()
        self.chomp_size = chomp_size

    def forward(self, x):
        return x[:, :, :-self.chomp_size].contiguous()

class TemporalBlock(nn.Module):
    """单个 TCN 块，包含两个带空洞和残差连接的卷积层"""
    def __init__(self, n_inputs, n_outputs, kernel_size, stride, dilation, padding, dropout=0.2):
        super(TemporalBlock, self).__init__()
        self.conv1 = nn.Conv1d(n_inputs, n_outputs, kernel_size,
                               stride=stride, padding=padding, dilation=dilation)
        self.chomp1 = Chomp1d(padding)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)

        self.conv2 = nn.Conv1d(n_outputs, n_outputs, kernel_size,
                               stride=stride, padding=padding, dilation=dilation)
        self.chomp2 = Chomp1d(padding)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)

        self.net = nn.Sequential(self.conv1, self.chomp1, self.relu1, self.dropout1,
                                 self.conv2, self.chomp2, self.relu2, self.dropout2)
        # 如果输入输出通道不同，需要一个 1x1 卷积进行下采样以匹配维度
        self.downsample = nn.Conv1d(n_inputs, n_outputs, 1) if n_inputs != n_outputs else None
        self.relu = nn.ReLU()
        self.init_weights()

    def init_weights(self):
        # 权重初始化
        self.conv1.weight.data.normal_(0, 0.01)
        self.conv2.weight.data.normal_(0, 0.01)
        if self.downsample is not None:
            self.downsample.weight.data.normal_(0, 0.01)

    def forward(self, x):
        out = self.net(x)
        res = x if self.downsample is None else self.downsample(x)
        return self.relu(out + res)

class TemporalConvNet(nn.Module):
    """时间卷积网络"""
    def __init__(self, num_inputs, num_channels, kernel_size=2, dropout=0.2):
        super(TemporalConvNet, self).__init__()
        layers = []
        num_levels = len(num_channels)
        for i in range(num_levels):
            dilation_size = 2 ** i # 空洞因子指数增长
            in_channels = num_inputs if i == 0 else num_channels[i-1]
            out_channels = num_channels[i]
            layers += [TemporalBlock(in_channels, out_channels, kernel_size, stride=1, dilation=dilation_size,
                                     padding=(kernel_size-1) * dilation_size, dropout=dropout)]

        self.network = nn.Sequential(*layers)

    def forward(self, x):
        """
        Args:
            x: 输入张量，形状为 [批次大小, 特征维度, 序列长度] (B, C, T)
        Returns:
            TCN 输出张量，形状为 [批次大小, 输出通道数, 序列长度] (B, C_out, T)
        """
        return self.network(x)

# --- 图像特征提取器 ---
class ImageFeatureExtractor(nn.Module):
    def __init__(self, convnext_model_name='convnext_tiny', pretrained=True, embed_dim=256, num_attn_heads=8, num_1d_conv_layers=2, dropout=0.1):
        super().__init__()
        self.embed_dim = embed_dim

        # 1. ConvNeXt Backbone
        self.convnext = timm.create_model(convnext_model_name, pretrained=pretrained, features_only=True, out_indices=[3]) # 使用最后一个阶段的输出
        # 获取 ConvNeXt 输出的通道数
        dummy_input = torch.randn(1, 3, 224, 224) # 假设输入尺寸
        dummy_features = self.convnext(dummy_input)
        self.convnext_out_dim = dummy_features[0].shape[1]

        # 2. 全局平均池化 (在时间维度上应用)
        self.gap = nn.AdaptiveAvgPool2d((1, 1))

        # 3. 线性投影到 embed_dim (可选，如果 C' != embed_dim)
        if self.convnext_out_dim != embed_dim:
             self.projection = nn.Linear(self.convnext_out_dim, embed_dim)
        else:
             self.projection = nn.Identity()

        # 4. 位置编码
        self.pos_encoder = PositionalEncoding(embed_dim)

        # 5. Transformer 块 (LN -> MHA -> Add -> LN)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.mha = nn.MultiheadAttention(embed_dim, num_attn_heads, dropout=dropout, batch_first=True) # 使用 batch_first=True
        self.norm2 = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)

        # 6. 多层 1D 卷积
        conv1d_layers = []
        in_channels = embed_dim
        for _ in range(num_1d_conv_layers):
            # 保持维度不变，只做特征提炼
            conv1d_layers.append(nn.Conv1d(in_channels, embed_dim, kernel_size=3, padding=1))
            conv1d_layers.append(nn.ReLU())
            # 可以选择性加入 BatchNorm1d 或 LayerNorm
            in_channels = embed_dim # 后续层的输入通道是 embed_dim
        self.multi_1d_conv = nn.Sequential(*conv1d_layers)
        self.final_dim = embed_dim # 最终输出维度

    def forward(self, x):
        """
        Args:
            x: 输入张量 [B, T, C, H, W]
        Returns:
            输出特征张量 [B, T, D_img]
        """
        B, T, C, H, W = x.shape
        x = x.view(B * T, C, H, W) # 合并 B 和 T 维度以输入 ConvNeXt

        # ConvNeXt 特征提取
        features = self.convnext(x)[0] # 获取指定阶段的输出 [B*T, C', H', W']

        # 全局平均池化
        pooled_features = self.gap(features).view(B * T, -1) # [B*T, C']

        # 投影 (如果需要)
        projected_features = self.projection(pooled_features) # [B*T, embed_dim]

        # 重塑形状并添加位置编码
        seq_features = projected_features.view(B, T, self.embed_dim) # [B, T, embed_dim]

        # PyTorch MHA 和 PositionalEncoding 需要 (T, B, C) 或 (B, T, C)
        # 如果使用 batch_first=True, MHA 输入是 (B, T, C)
        # PositionalEncoding 内部处理 (T, B, C)，所以我们需要调整
        seq_features_tbc = seq_features.permute(1, 0, 2) # [T, B, embed_dim]
        seq_features_pe = self.pos_encoder(seq_features_tbc) # [T, B, embed_dim]
        seq_features_pe_btc = seq_features_pe.permute(1, 0, 2) # [B, T, embed_dim]

        # Transformer 块
        x_norm1 = self.norm1(seq_features_pe_btc)
        # MHA 输入: query, key, value 都是 x_norm1
        attn_output, _ = self.mha(x_norm1, x_norm1, x_norm1)
        # 残差连接: 输入是添加了 PE 的特征
        x_res = seq_features_pe_btc + self.dropout(attn_output)
        x_addnorm = self.norm2(x_res) # [B, T, embed_dim]

        # 多层 1D 卷积
        # Conv1d 需要 [B, C, T] 格式
        x_addnorm_bct = x_addnorm.permute(0, 2, 1) # [B, embed_dim, T]
        f_img = self.multi_1d_conv(x_addnorm_bct) # [B, embed_dim, T]
        f_img = f_img.permute(0, 2, 1) # 转回 [B, T, embed_dim]

        return f_img

# --- 时间序列 (电流功率) 特征提取器 ---
class TimeSeriesFeatureExtractor(nn.Module):
    def __init__(self, input_dim=2, proj_dim=64, tcn_channels=[128, 256], tcn_kernel_size=3, embed_dim=256, num_attn_heads=8, dropout=0.1):
        super().__init__()
        self.embed_dim = embed_dim

        # 1. 初步特征映射 (使用 Conv1D)
        self.conv1d_proj = nn.Conv1d(input_dim, proj_dim, kernel_size=1) # kernel_size=1 类似 FC
        self.relu_proj = nn.ReLU()

        # 2. TCN
        # TCN 输入需要 [B, C, T], 输出 [B, C_out, T]
        self.tcn = TemporalConvNet(proj_dim, tcn_channels, kernel_size=tcn_kernel_size, dropout=dropout)
        tcn_out_dim = tcn_channels[-1]

        # 3. 投影到 embed_dim (如果 TCN 输出维度不同)
        if tcn_out_dim != embed_dim:
            self.tcn_proj = nn.Conv1d(tcn_out_dim, embed_dim, 1) # 使用 1x1 Conv 调整通道
        else:
            self.tcn_proj = nn.Identity()

        # 4. 位置编码
        self.pos_encoder = PositionalEncoding(embed_dim)

        # 5. Transformer 块 (LN -> MHA -> Add -> LN)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.mha = nn.MultiheadAttention(embed_dim, num_attn_heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)
        self.final_dim = embed_dim

    def forward(self, x):
        """
        Args:
            x: 输入张量 [B, T, C_in] (C_in=2)
        Returns:
            输出特征张量 [B, T, D_cp]
        """
        # 调整维度以适应 Conv1d: [B, C_in, T]
        x = x.permute(0, 2, 1)

        # 初步映射
        x_proj = self.relu_proj(self.conv1d_proj(x)) # [B, proj_dim, T]

        # TCN
        x_tcn = self.tcn(x_proj) # [B, tcn_out_dim, T]

        # 投影到 embed_dim
        x_tcn_proj = self.tcn_proj(x_tcn) # [B, embed_dim, T]

        # 调整维度以添加 PE 和输入 Transformer
        seq_features = x_tcn_proj.permute(0, 2, 1) # [B, T, embed_dim]

        # 添加位置编码
        seq_features_tbc = seq_features.permute(1, 0, 2) # [T, B, embed_dim]
        seq_features_pe = self.pos_encoder(seq_features_tbc) # [T, B, embed_dim]
        seq_features_pe_btc = seq_features_pe.permute(1, 0, 2) # [B, T, embed_dim]

        # Transformer 块
        x_norm1 = self.norm1(seq_features_pe_btc)
        attn_output, _ = self.mha(x_norm1, x_norm1, x_norm1)
        # 残差连接: 输入是添加了 PE 的特征
        x_res = seq_features_pe_btc + self.dropout(attn_output)
        f_cp = self.norm2(x_res) # [B, T, embed_dim]

        return f_cp

# --- 多模态融合模块 ---
class MultiModalFusion(nn.Module):
    def __init__(self, img_dim, cp_dim, fused_dim, gate_dim, num_attn_heads=8, dropout=0.1):
        super().__init__()
        self.img_dim = img_dim
        self.cp_dim = cp_dim
        self.fused_dim = fused_dim # 交叉注意力之前的投影维度
        self.gate_dim = gate_dim   # 门控融合的输出维度

        # 1. 特征投影 FC 层
        self.fc_img = nn.Linear(img_dim, fused_dim)
        self.fc_cp = nn.Linear(cp_dim, fused_dim)

        # 2. 交叉注意力层
        # 注意力维度 d_k 通常是 fused_dim / num_attn_heads
        self.cross_attn_img_to_cp = nn.MultiheadAttention(fused_dim, num_attn_heads, dropout=dropout, batch_first=True)
        self.cross_attn_cp_to_img = nn.MultiheadAttention(fused_dim, num_attn_heads, dropout=dropout, batch_first=True)

        # 3. 动态门控融合层
        # 两个 tanh 变换的全连接层
        self.fc_h1 = nn.Linear(fused_dim, gate_dim) # MHA 输出维度是 fused_dim
        self.fc_h2 = nn.Linear(fused_dim, gate_dim)
        # 计算门控系数 z 的全连接层 (输入是两个交叉注意力输出的拼接)
        self.fc_z = nn.Linear(fused_dim * 2, gate_dim)

        self.dropout_cross_attn = nn.Dropout(dropout) # 可选的 Dropout

    def forward(self, f_img, f_cp):
        """
        Args:
            f_img: 图像特征 [B, T, D_img]
            f_cp: 电流功率特征 [B, T, D_cp]
        Returns:
            融合后的特征 [B, T, D_gate]
        """
        # 特征投影
        f_img_proj = self.fc_img(f_img) # [B, T, fused_dim]
        f_cp_proj = self.fc_cp(f_cp)   # [B, T, fused_dim]

        # 交叉注意力
        # 图像特征关注时序特征 (Q=img, K=cp, V=cp)
        o_img_to_cp, _ = self.cross_attn_img_to_cp(f_img_proj, f_cp_proj, f_cp_proj)
        o_img_to_cp = self.dropout_cross_attn(o_img_to_cp) # [B, T, fused_dim]

        # 时序特征关注图像特征 (Q=cp, K=img, V=img)
        o_cp_to_img, _ = self.cross_attn_cp_to_img(f_cp_proj, f_img_proj, f_img_proj)
        o_cp_to_img = self.dropout_cross_attn(o_cp_to_img) # [B, T, fused_dim]

        # 动态门控融合
        h_img_to_cp = torch.tanh(self.fc_h1(o_img_to_cp)) # [B, T, gate_dim]
        h_cp_to_img = torch.tanh(self.fc_h2(o_cp_to_img)) # [B, T, gate_dim]

        # 计算门控系数 z
        z_input = torch.cat([o_img_to_cp, o_cp_to_img], dim=-1) # [B, T, fused_dim * 2]
        z = torch.sigmoid(self.fc_z(z_input)) # [B, T, gate_dim]

        # 加权融合
        f_fused = z * h_img_to_cp + (1 - z) * h_cp_to_img # [B, T, gate_dim]

        return f_fused

# --- 预测头 ---
class PredictionHead(nn.Module):
    def __init__(self, input_dim, gru_hidden_dim, num_gru_layers=1, num_error_axes=1, dropout=0.1):
        super().__init__()
        # GRU 层
        # 注意：GRU 输入需要 [T, B, C] 或 [B, T, C] (如果 batch_first=True)
        self.gru = nn.GRU(input_dim, gru_hidden_dim, num_layers=num_gru_layers,
                          batch_first=True, dropout=dropout if num_gru_layers > 1 else 0) # 只有多层 GRU 时中间层才加 dropout

        # 最终的全连接层
        self.fc_out = nn.Linear(gru_hidden_dim, num_error_axes)

    def forward(self, x):
        """
        Args:
            x: 融合后的特征 [B, T, D_gate]
        Returns:
            预测的热误差 [B, N_error]
        """
        # GRU 处理序列
        # gru_out 包含所有时间步的隐藏状态 [B, T, D_gru]
        # h_n 是最后一个时间步的隐藏状态 [num_layers, B, D_gru]
        gru_out, h_n = self.gru(x)

        # 取最后一个时间步的最后一个 GRU 层的隐藏状态
        # h_n 的形状是 [num_layers, B, D_gru], 我们需要最后一个 layer 的输出
        last_hidden_state = h_n[-1, :, :] # [B, D_gru]

        # 或者，也可以取 gru_out 的最后一个时间步输出
        # last_hidden_state = gru_out[:, -1, :] # [B, D_gru]

        # 通过 FC 层进行最终预测
        prediction = self.fc_out(last_hidden_state) # [B, N_error]

        return prediction

# --- 完整模型 ---
class ThermalErrorPredictor(nn.Module):
    def __init__(self, img_config, cp_config, fusion_config, pred_config):
        super().__init__()
        self.image_feature_extractor = ImageFeatureExtractor(**img_config)
        self.timeseries_feature_extractor = TimeSeriesFeatureExtractor(**cp_config)
        # 确保融合层的输入维度匹配
        fusion_config['img_dim'] = self.image_feature_extractor.final_dim
        fusion_config['cp_dim'] = self.timeseries_feature_extractor.final_dim
        self.multi_modal_fusion = MultiModalFusion(**fusion_config)
        # 确保预测头的输入维度匹配
        pred_config['input_dim'] = self.multi_modal_fusion.gate_dim
        self.prediction_head = PredictionHead(**pred_config)

    def forward(self, x_img, x_cp):
        """
        Args:
            x_img: 红外图像序列 [B, T, C, H, W]
            x_cp: 电流功率序列 [B, T, 2]
        Returns:
            prediction: 预测的热误差 [B, N_error]
            f_img: 图像特征 [B, T, D_img] (用于 DTL)
            f_cp: 电流功率特征 [B, T, D_cp] (用于 DTL)
            f_fused: 融合特征 [B, T, D_gate] (用于 DTL)
        """
        f_img = self.image_feature_extractor(x_img)
        f_cp = self.timeseries_feature_extractor(x_cp)
        f_fused = self.multi_modal_fusion(f_img, f_cp)
        prediction = self.prediction_head(f_fused)

        # 返回预测结果和用于 DTL 的中间特征
        return prediction, f_img, f_cp, f_fused

# --- 损失计算 ---
def calculate_loss(y_pred_s, y_true_s, f_fused_s, f_fused_t, lambda_dtw=0.1, sdtw_gamma=0.1):
    """
    计算总损失 L_total = L_task + lambda * L_DTW
    Args:
        y_pred_s: 源域预测值 [B_S, N_error]
        y_true_s: 源域真实值 [B_S, N_error]
        f_fused_s: 源域融合特征 [B_S, T, D_gate]
        f_fused_t: 目标域融合特征 [B_T, T, D_gate]
        lambda_dtw: DTW 损失的权重
        sdtw_gamma: SoftDTW 的平滑参数
    Returns:
        total_loss: 总损失
        task_loss: 任务损失 (MSE)
        dtw_loss: SoftDTW 域损失
    """
    # 1. 任务损失 (MSE)
    task_loss = F.mse_loss(y_pred_s, y_true_s)

    # 2. DTW 域损失 (使用 SoftDTW)
    # SoftDTW 需要 [B, T, D] 格式
    # 注意：SoftDTW 计算的是 batch 内所有样本对之间的距离矩阵的某种平均
    # 这里我们简化处理，假设 B_S = B_T = B，并计算对应样本的 SoftDTW
    # 或者，更标准的做法是计算所有源域和目标域样本间的距离矩阵，然后使用 MMD-like 的方式处理
    # 为了简单起见，我们计算批次内对应样本的平均 SoftDTW
    # 需要确保 B_S == B_T
    if f_fused_s.shape[0] != f_fused_t.shape[0]:
         # 如果 batch size 不同，无法直接计算对应样本的 DTW
         # 可以采取其他策略，例如随机采样配对，或者使用其他 DTL 损失
         # 这里暂时返回 0，实际应用中需要处理这种情况
         print("Warning: Source and target batch sizes differ for DTW loss calculation. Returning DTW loss as 0.")
         dtw_loss = torch.tensor(0.0, device=f_fused_s.device) # 返回一个 tensor
    else:
        sdtw = SoftDTW(use_cuda=f_fused_s.is_cuda, gamma=sdtw_gamma)
        # SoftDTW 计算的是每对序列的距离，我们需要求平均
        # sdtw(f_fused_s, f_fused_t) 返回一个 [B] 大小的张量
        dtw_loss = sdtw(f_fused_s, f_fused_t).mean()

    # 3. 总损失
    total_loss = task_loss + lambda_dtw * dtw_loss

    return total_loss, task_loss, dtw_loss


# --- 示例用法 ---
if __name__ == '__main__':
    # 定义配置参数 (需要根据实际情况调整)
    IMG_CONFIG = {
        'convnext_model_name': 'convnext_tiny',
        'pretrained': False, # 实际使用时建议设为 True
        'embed_dim': 256,
        'num_attn_heads': 4,
        'num_1d_conv_layers': 2,
        'dropout': 0.1
    }
    CP_CONFIG = {
        'input_dim': 2,
        'proj_dim': 64,
        'tcn_channels': [128, 128], # TCN 输出通道数
        'tcn_kernel_size': 3,
        'embed_dim': 256, # 需要与图像分支对齐
        'num_attn_heads': 4,
        'dropout': 0.1
    }
    FUSION_CONFIG = {
        # img_dim 和 cp_dim 会在模型初始化时自动设置
        'fused_dim': 256, # 交叉注意力之前的投影维度
        'gate_dim': 256,  # 门控融合输出维度
        'num_attn_heads': 4,
        'dropout': 0.1
    }
    PRED_CONFIG = {
        # input_dim 会在模型初始化时自动设置
        'gru_hidden_dim': 128,
        'num_gru_layers': 1,
        'num_error_axes': 1, # 假设预测单个误差值
        'dropout': 0.1
    }

    # 实例化模型
    model = ThermalErrorPredictor(IMG_CONFIG, CP_CONFIG, FUSION_CONFIG, PRED_CONFIG)

    # 创建模拟输入数据
    B_S = 4 # 源域批次大小
    B_T = 4 # 目标域批次大小
    T = 50  # 时间序列长度
    H, W = 224, 224 # 图像尺寸
    C = 3 # 图像通道数 (ConvNeXt 通常需要 3 通道)

    # 模拟源域数据
    dummy_img_s = torch.randn(B_S, T, C, H, W)
    dummy_cp_s = torch.randn(B_S, T, 2)
    dummy_y_s = torch.randn(B_S, PRED_CONFIG['num_error_axes'])

    # 模拟目标域数据 (只有输入，没有标签)
    dummy_img_t = torch.randn(B_T, T, C, H, W)
    dummy_cp_t = torch.randn(B_T, T, 2)

    # 将模型移到 GPU (如果可用)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    dummy_img_s = dummy_img_s.to(device)
    dummy_cp_s = dummy_cp_s.to(device)
    dummy_y_s = dummy_y_s.to(device)
    dummy_img_t = dummy_img_t.to(device)
    dummy_cp_t = dummy_cp_t.to(device)


    # 模型前向传播 (源域)
    # 通常在训练时，源域和目标域数据会一起或交替输入
    # 这里为了演示，分开调用
    model.train() # 设置为训练模式
    y_pred_s, f_img_s, f_cp_s, f_fused_s = model(dummy_img_s, dummy_cp_s)

    # 模型前向传播 (目标域 - 获取特征用于 DTL)
    # 在实际训练中，可能不需要目标域的预测值
    model.eval() # 或者保持 train() 模式，取决于 DTL 实现
    with torch.no_grad(): # 通常计算 DTL 特征时不需要梯度
         _, f_img_t, f_cp_t, f_fused_t = model(dummy_img_t, dummy_cp_t)


    # 计算损失
    total_loss, task_loss, dtw_loss = calculate_loss(y_pred_s, dummy_y_s, f_fused_s, f_fused_t, lambda_dtw=0.1, sdtw_gamma=0.1)

    print(f"模型输出预测形状 (源域): {y_pred_s.shape}")
    print(f"融合特征形状 (源域): {f_fused_s.shape}")
    print(f"融合特征形状 (目标域): {f_fused_t.shape}")
    print("-" * 20)
    print(f"任务损失 (MSE): {task_loss.item():.4f}")
    print(f"DTW 域损失: {dtw_loss.item():.4f}")
    print(f"总损失: {total_loss.item():.4f}")

    # 接下来可以进行反向传播和优化器步骤
    # optimizer.zero_grad()
    # total_loss.backward()
    # optimizer.step()